# 🔍 Exploratory Data Analysis — SwiftEats Platform

High-level exploration of orders, customers, restaurants, and delivery patterns.

## 1. Setup & Connection

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER','food_user')}:{os.getenv('DB_PASSWORD','')}"
    f"@{os.getenv('DB_HOST','localhost')}:{os.getenv('DB_PORT','5432')}"
    f"/{os.getenv('DB_NAME','food_delivery')}"
)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
print("Connected to SwiftEats database ✓")

## 2. Platform Overview

In [ ]:
with engine.connect() as conn:
    kpi = pd.read_sql(text('''
        SELECT
            COUNT(*) FILTER (WHERE order_status = 'Delivered')  AS delivered,
            COUNT(*) FILTER (WHERE order_status = 'Cancelled')  AS cancelled,
            COUNT(*)                                            AS total_orders,
            ROUND(SUM(total_amount) FILTER (WHERE order_status = 'Delivered')::numeric,0) AS gmv,
            COUNT(DISTINCT customer_id)                         AS customers,
            COUNT(DISTINCT restaurant_id)                       AS restaurants
        FROM orders
    '''), conn).iloc[0]

print("=" * 45)
print(f"  Total Orders:      {int(kpi['total_orders']):>10,}")
print(f"  Delivered:         {int(kpi['delivered']):>10,}")
print(f"  Cancelled:         {int(kpi['cancelled']):>10,}")
print(f"  Platform GMV:      ₹{float(kpi['gmv']):>10,.0f}")
print(f"  Unique Customers:  {int(kpi['customers']):>10,}")
print(f"  Active Restaurants:{int(kpi['restaurants']):>10,}")
print("=" * 45)

## 3. Order Volume Trend

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT order_date, COUNT(*) AS orders, ROUND(SUM(total_amount)::numeric,2) AS revenue
        FROM orders WHERE order_status = 'Delivered'
          AND order_date >= CURRENT_DATE - 90
        GROUP BY order_date ORDER BY order_date
    '''), conn)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))
ax1.plot(df['order_date'], df['orders'], linewidth=1.5, color='steelblue')
ax1.fill_between(df['order_date'], df['orders'], alpha=0.15, color='steelblue')
ax1.set_title('Daily Delivered Orders — Last 90 Days')
ax1.set_ylabel('Orders')

# 7-day MA
df['ma7'] = df['orders'].rolling(7).mean()
ax1.plot(df['order_date'], df['ma7'], color='orange', linewidth=2, label='7-Day MA')
ax1.legend()

ax2.plot(df['order_date'], df['revenue'], color='green', linewidth=1.5)
ax2.fill_between(df['order_date'], df['revenue'], alpha=0.15, color='green')
ax2.set_title('Daily Revenue (₹) — Last 90 Days')
ax2.set_ylabel('Revenue (₹)')
ax2.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 4. Order Status Distribution

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT order_status, COUNT(*) AS count,
               ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER()::numeric,1) AS pct
        FROM orders GROUP BY order_status ORDER BY count DESC
    '''), conn)

colors = {'Delivered': '#27AE60', 'Cancelled': '#E74C3C', 'On The Way': '#F39C12',
          'Preparing': '#3498DB', 'Placed': '#95A5A6', 'Confirmed': '#2ECC71'}
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(df['order_status'], df['count'],
               color=[colors.get(s, '#BDC3C7') for s in df['order_status']])
for bar, pct in zip(bars, df['pct']):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
            f' {pct}%', va='center', fontsize=10)
ax.set_title('Order Status Distribution')
ax.set_xlabel('Number of Orders')
plt.tight_layout()
plt.show()

## 5. Peak Demand Heatmap

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT day_of_week, day_name, hour_of_day,
               order_count, demand_index
        FROM mv_hourly_demand
    '''), conn)

pivot = df.pivot_table(index='day_of_week', columns='hour_of_day',
                        values='demand_index', aggfunc='mean')
pivot.index = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Demand Index'},
            annot=pivot.round(1), fmt='.1f', annot_kws={'size': 7})
ax.set_title('Order Demand Heatmap — Day × Hour (1.0 = average, >2.0 = rush)', fontsize=13)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.show()
print("Dinner peak (19-22h) and weekends dominate demand")

## 6. Cuisine Performance

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT cuisine_type, COUNT(o.order_id) AS orders,
               ROUND(SUM(o.total_amount)::numeric,0) AS revenue,
               ROUND(AVG(o.total_amount)::numeric,2) AS aov
        FROM orders o JOIN restaurants r ON o.restaurant_id = r.restaurant_id
        WHERE o.order_status = 'Delivered'
        GROUP BY r.cuisine_type ORDER BY revenue DESC LIMIT 12
    '''), conn)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(df['cuisine_type'], df['revenue'], color=sns.color_palette("Set2", len(df)))
ax.set_xticklabels(df['cuisine_type'], rotation=45, ha='right')
ax.set_title('Revenue by Cuisine Type (Top 12)')
ax.set_ylabel('Total Revenue (₹)')
for bar, aov in zip(bars, df['aov']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ax.get_ylim()[1]*0.005,
            f'₹{aov:.0f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()